# NLP Task 4: Fine-Tuning BERT on IMDB Dataset


In [ ]:
!pip install transformers datasets scikit-learn

In [ ]:
import numpy as np
from datasets import load_dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

In [ ]:
# Load dataset
dataset = load_dataset('imdb')

In [ ]:
# Preprocessing
def clean_text(example):
    text = example['text'].lower()
    text = ' '.join(text.split())
    return {'text': text}

dataset = dataset.map(clean_text)

In [ ]:
# Split data
split = dataset['train'].train_test_split(test_size=0.1)

dataset = DatasetDict({
    'train': split['train'],
    'validation': split['test'],
    'test': dataset['test']
})

In [ ]:
# Tokenization
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

def tokenize(example):
    return tokenizer(example['text'], padding='max_length', truncation=True)

tokenized_dataset = dataset.map(tokenize, batched=True)
tokenized_dataset = tokenized_dataset.remove_columns(['text'])
tokenized_dataset.set_format('torch')

In [ ]:
# Reduce dataset for faster training
small_train = tokenized_dataset['train'].select(range(1000))
small_val = tokenized_dataset['validation'].select(range(200))
small_test = tokenized_dataset['test'].select(range(200))

In [ ]:
# Load model
model = AutoModelForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)

In [ ]:
# Metrics
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    acc = accuracy_score(labels, preds)

    return {
        'accuracy': acc,
        'precision': precision,
        'recall': recall,
        'f1': f1
    }

In [ ]:
# Training arguments
training_args = TrainingArguments(
    output_dir='./results',
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=1
)

In [ ]:
# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train,
    eval_dataset=small_val,
    compute_metrics=compute_metrics
)

In [ ]:
# Train model
trainer.train()

In [ ]:
# Evaluation
results = trainer.evaluate()
print(results)

In [ ]:
# Confusion Matrix
preds_output = trainer.predict(small_test)

y_pred = np.argmax(preds_output.predictions, axis=1)
y_true = preds_output.label_ids

cm = confusion_matrix(y_true, y_pred)
print('Confusion Matrix:\n', cm)